# Alpha-u single-shot broad benchmark

Clean driver for the alpha-parameterization simplification experiment. It executes the committed broad single-shot benchmark and changes only the nonlinear identifier to `identify_nonlinear_unbounded`, adds the identifier to implementation hashing, and uses the short internal namespace `alpha_u_ss` to avoid Windows path-length failures.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

# This is intentionally a thin ablation driver.  The committed broad benchmark
# remains the source of truth; only the alpha identifier and filesystem/cache
# namespace are changed here.
HERE = Path.cwd()

BASE_CANDIDATES = [
    HERE / "experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed_BACKFILL_FIXED.ipynb",
    HERE / "experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed.ipynb",
]
BASE_NOTEBOOK = next((p for p in BASE_CANDIDATES if p.exists()), None)
if BASE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Could not find the committed broad single-shot base notebook. Looked for:\n"
        + "\n".join(str(p) for p in BASE_CANDIDATES)
    )

print("Alpha-u single-shot base:", BASE_NOTEBOOK.name)

base_nb = json.loads(BASE_NOTEBOOK.read_text(encoding="utf-8"))

patched_import = False
patched_study = False
patched_results = False
patched_source_hash = False
patched_scientific_config = False

patched_cells: list[tuple[int, str]] = []

for cell_index, cell in enumerate(base_nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    # 1) Swap only the identifier used by the existing single-shot runner.
    anchor = (
        "import opinion_dynamics.experiments.online_single_shot "
        "as online_single_shot_module\n"
    )
    if anchor in src and "identify_nonlinear_unbounded" not in src:
        src = src.replace(
            anchor,
            anchor
            + "import opinion_dynamics.identify_nonlinear_unbounded as identifier_module\n"
            + "online_single_shot_module.GraphIdentifierEnv = identifier_module.GraphIdentifierEnv\n"
            + "online_single_shot_module.train_graph_identifier = identifier_module.train_graph_identifier\n"
            + "online_single_shot_module.pairs_from_intermediate = identifier_module.pairs_from_intermediate\n",
            1,
        )
        patched_import = True

        print_anchor = (
            'print("online_single_shot:", '
            'inspect.getsourcefile(online_single_shot_module))'
        )
        if print_anchor in src and 'print("identifier:"' not in src:
            src = src.replace(
                print_anchor,
                print_anchor
                + '\nprint("identifier:", inspect.getsourcefile(identifier_module))',
                1,
            )

    # 2) Short internal namespace: avoids Windows MAX_PATH issues.
    new_src, n = re.subn(
        r'STUDY_NAME\s*=\s*"[^"]+"',
        'STUDY_NAME = "alpha_u_ss"',
        src,
        count=1,
    )
    if n:
        src = new_src
        patched_study = True

    new_src, _ = re.subn(
        r'PIPELINE_VERSION\s*=\s*"[^"]+"',
        'PIPELINE_VERSION = "2026-09-06-alpha-u-ss-v1"',
        src,
        count=1,
    )
    src = new_src

    for old_name in (
        "experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed_BACKFILL_FIXED",
        "experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed",
    ):
        if old_name in src:
            src = src.replace(old_name, "alpha_u_ss")
            patched_results = True

    # 3) Include the new identifier source in implementation hashing.
    if "SOURCE_HASHES = {" in src and '"identify_nonlinear_unbounded"' not in src:
        src = src.replace(
            "SOURCE_HASHES = {\n",
            'SOURCE_HASHES = {\n'
            '    "identify_nonlinear_unbounded": source_hash(identifier_module),\n',
            1,
        )
        patched_source_hash = True

    # 4) Make the alpha parameterization explicit in the scientific config.
    if "SCIENTIFIC_CONFIG = {" in src and '"alpha_parameterization"' not in src:
        src = src.replace(
            "SCIENTIFIC_CONFIG = {\n",
            'SCIENTIFIC_CONFIG = {\n'
            '    "alpha_parameterization": "softplus_over_log2_positive_unbounded",\n',
            1,
        )
        patched_scientific_config = True

    patched_cells.append((cell_index, src))

required = {
    "identifier import": patched_import,
    "short STUDY_NAME": patched_study,
    "short RESULTS_DIR": patched_results,
    "identifier source hash": patched_source_hash,
    "alpha scientific-config marker": patched_scientific_config,
}
missing = [name for name, ok in required.items() if not ok]
if missing:
    raise RuntimeError(
        "Refusing to execute: the committed base notebook did not match the "
        "expected broad-benchmark structure. Missing patches: " + ", ".join(missing)
    )

print("Alpha-u overrides validated:")
for name in required:
    print("  OK:", name)

# Execute the committed benchmark cells in one shared namespace, exactly like
# run_ipynb_cells.py, after the validated in-memory overrides above.
g = globals()
for cell_index, src in patched_cells:
    print(f"[base code cell {cell_index}]")
    exec(
        compile(src, f"{BASE_NOTEBOOK.name}:cell_{cell_index}", "exec"),
        g,
        g,
    )
